In [1]:
import pandas as pd

df = pd.read_csv("../data/marketing_AB.csv")
print(df.shape)
df.head()

(588101, 7)


,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [2]:
df = df.drop(columns="Unnamed: 0")
df.columns = ["user_id", "group", "converted", "total_ads", "most_ads_day", "most_ads_hour"]
df.dtypes

user_id           int64
group            object
converted          bool
total_ads         int64
most_ads_day     object
most_ads_hour     int64
dtype: object

In [3]:
duplicates = df["user_id"].duplicated().sum()
missing = df.isna().sum()
unique_groups = df["group"].unique()
unique_converted = df["converted"].unique()

print("Количество дублей user_id:", duplicates)
print("Количество пропусков по столбцам:\n", missing, sep = "")
print("Уникальные значения в столбце group:", unique_groups)
print("Уникальные значения в столбце converted:", unique_converted)

Количество дублей user_id: 0
Количество пропусков по столбцам:
user_id          0
group            0
converted        0
total_ads        0
most_ads_day     0
most_ads_hour    0
dtype: int64
Уникальные значения в столбце group: ['ad' 'psa']
Уникальные значения в столбце converted: [False  True]


In [4]:
import sqlite3

con = sqlite3.connect(":memory:")
df.to_sql("users", con, index = False)

query = """
SELECT
    "group",
    users,
    users * 1.0 / SUM(users) OVER () AS shares,
    conversions,
    conv_rate
FROM (
    SELECT
        "group",
        COUNT(*) AS users,
        SUM(converted) AS conversions,
        AVG(converted) AS conv_rate
    FROM users
    GROUP BY "group"
    )
"""
pd.read_sql(query, con)

,group,users,shares,conversions,conv_rate
0,ad,564577,0.96,14423,0.025547
1,psa,23524,0.04,420,0.017854


In [5]:
from scipy import stats

counts = df["group"].value_counts()
n_ad = counts["ad"]
n_psa = counts["psa"]
n_total = n_ad + n_psa

observed = [int(n_ad), int(n_psa)]
expected = [int(n_total) * 0.96, int(n_total) * 0.04]

chi2, p_value = stats.chisquare(observed, f_exp=expected)

print ("Наблюдаем:", observed)
print ("Ожидали:  ", expected)
print ("p_value:  ", p_value)

Наблюдаем: [564577, 23524]
Ожидали:   [564576.96, 23524.04]
p_value:   0.9997876225135022


In [6]:
from scipy.stats import norm

alpha = 0.05
power = 0.80

mask = (df["group"] == "psa")
psa = df[mask]
p_base = psa["converted"].mean()

z_alpha = norm.ppf(1 - alpha / 2)
z_beta = norm.ppf(power)

se = (p_base * (1 - p_base) * (1 / n_ad + 1 / n_psa)) ** 0.5
mde = (z_alpha + z_beta) * se

print("z_alpha:", round(z_alpha, 3), " z_beta:", round(z_beta, 3))
print("MDE, п.п.:", round(mde * 100, 2))
print("MDE, отн. %:", round(mde / p_base * 100, 1))

z_alpha: 1.96  z_beta: 0.842
MDE, п.п.: 0.25
MDE, отн. %: 13.8


In [7]:
import math

delta = 0.003
z_sum = z_alpha + z_beta

def sample_size(share_ad, share_psa):
    n_total = z_sum ** 2 * p_base * (1 - p_base) * (1 / share_ad + 1 / share_psa) / delta ** 2
    return math.ceil(n_total * share_ad), math.ceil(n_total * share_psa)

ad_50, psa_50 = sample_size(0.5, 0.5)
ad_96, psa_96 = sample_size(0.96, 0.04)

print("50/50:", ad_50, "+", psa_50, "=", ad_50 + psa_50)
print("96/4: ", ad_96, "+", psa_96, "=", ad_96 + psa_96)

50/50: 30586 + 30586 = 61172
96/4:  382314 + 15930 = 398244
